# Job Scout — Phase 2: Tailoring + the Evaluation Stack

**Build → Evaluate → Self-Improve.** This is Phase 2: the agent now *prepares applications*
(a corpus-grounded cover letter + a tailored LaTeX CV), and we build the evaluation stack
around it. The spine of this part: **verifiable checks vs unverifiable judgments**.

> The human applies. The agent never submits applications.

Run top to bottom. Cells that spend money or need keys say so.

## 1. Environment check

In [ ]:
import importlib.metadata as m

for pkg in ["langgraph", "langchain", "opik", "gradio", "jinja2", "pyyaml"]:
    print(f"{pkg:12s} {m.version(pkg)}")

## 2. Service / key checks

Phase 2 needs an LLM key for tailoring and an Opik key for the eval stack.
`TAVILY_API_KEY` (company research) and the `tectonic` binary (PDF compile) are optional —
everything degrades gracefully without them.

In [ ]:
import shutil
from job_scout.config import get_settings

settings = get_settings()
checks = {
    "LLM key (OPENAI_API_KEY)": bool(settings.openai_api_key.get_secret_value()),
    "Opik tracing": settings.has_opik,
    "Tavily research (optional)": settings.has_tavily,
    "tectonic binary (optional)": shutil.which("tectonic") is not None,
}
for name, ok in checks.items():
    print(f"{'✓' if ok else '—'} {name}")

## 3. The CandidateCorpus — the only permissible content source

Tailoring may only **select and reword** items from the corpus: the CV text, plus an
*optional* official LinkedIn data export (Settings → "Get a copy of your data").
No API clients, no scrapers — and real user exports are never committed or attached to traces.
The committed ZIPs are synthetic fixtures.

In [ ]:
from pathlib import Path
from job_scout.corpus import build_corpus
from job_scout.tools.cv_reader import extract_cv_text

cv_path = Path("../data/fixture_cvs/senior_mle_eu.pdf")
cv_text = extract_cv_text(cv_path)

cv_only = build_corpus(cv_text)
merged = build_corpus(cv_text, "../data/fixture_linkedin/senior_mle_eu_li_export.zip")
print(f"CV only: {len(cv_only.items)} items · CV + LinkedIn: {len(merged.items)} items")
for item in merged.items[:6] + merged.items[-4:]:
    print(f"  [{item.id}] ({item.source}/{item.kind}) {item.text[:60]}")

## 4. Invocation A — the job search

Same graph as Phase 1, but entry now goes through `route_entry`. We pass
`selected_job_id=None` **explicitly** — remember that; it matters in §7.

*(Costs a few cents with API models; 1–3 minutes.)*

In [ ]:
from uuid import uuid4
from job_scout.profile import extract_profile
from job_scout.runner import stream_search

thread_id = str(uuid4())
profile = extract_profile(cv_text, thread_id=thread_id, tags=["phase-2", "notebook"])

result = None
for kind, payload in stream_search(profile, cv_text=cv_text, cv_path=str(cv_path),
                                   thread_id=thread_id, tags=["phase-2", "notebook"]):
    if kind == "status":
        print("·", payload)
    else:
        result = payload
print(f"\n{result.n_jobs_ranked} jobs ranked · ${result.cost_usd:.4f} · {result.latency_s}s")
for r in result.ranked_jobs[:5]:
    print(f"  {r.fit_score:3d}  {r.job.title} — {r.job.company}")

## 5. Invocation B — tailoring on the SAME thread

The headline lesson: this invocation passes **only `selected_job_id`**. Profile, ranked
jobs, and CV text come from the thread's checkpoint. Watch the streamed node names —
**no fetch, no rank** — and check the trace in Opik: the tailor trace contains only
`tailor` and `validate_tailoring` spans.

In [ ]:
from job_scout.runner import stream_tailor

job_id = result.ranked_jobs[0].job.job_id
tailor_result = None
for kind, payload in stream_tailor(thread_id=thread_id, selected_job_id=job_id,
                                   tags=["phase-2", "notebook", "tailor"]):
    if kind == "status":
        print("·", payload)
    else:
        tailor_result = payload

pack = tailor_result.pack
print(f"\nfabrication flags: {tailor_result.fabrication_flags}",
      f"of {tailor_result.fabrication_report.claims_checked} claims checked")
print("\n--- cover letter (first 400 chars) ---\n", pack.cover_letter[:400])
print("\n--- honesty note ---\n", pack.honesty_note)

## 6. The fabrication validator — a verifiable check

Deterministic, zero LLM calls: every bullet must resolve its `corpus_ref` and stay close
to it; skills must come from the corpus; factual cover-letter sentences must trace to the
corpus (or the research notes). It **logs, never retries** — flags are eval material.

Here is a deliberately fabricated pack failing loudly:

In [ ]:
from job_scout.graph.schemas import CVContent, ExperienceEntry, TailoredBullet, TailoringPack
from job_scout.validation import validate_pack

fabricated = TailoringPack(
    cv=CVContent(
        headline="Chief AI Officer",
        summary="Visionary leader.",
        experience=[ExperienceEntry(role="ML Engineer", company="DeepMobility GmbH", bullets=[
            TailoredBullet(text="Single-handedly built AGI, saving $4B annually", corpus_ref="cv-bullet-002"),
            TailoredBullet(text="Led SpaceX Mars landing software", corpus_ref="does-not-exist"),
        ])],
        skills=["Python", "Quantum Computing"],
    ),
    cover_letter="Dear team,\nI increased revenue by 900% at Google Brain in 2024.\nBest, L.",
)
report = validate_pack(fabricated, merged)
print(f"{report.flags} of {report.claims_checked} claims flagged:")
for f in report.flagged:
    print(f"  [{f.where}] {f.reason}")

## 7. The stale-state bug — why the runner nulls `selected_job_id`

With one shared checkpointer, thread state **persists across invocations**. If a new
search on a used thread *omits* `selected_job_id`, the checkpointed value from the last
tailoring run is still there — and the entry router sends your fresh search into `tailor`
against a stale job. The production runner always passes `selected_job_id=None`; here we
reproduce the bug on purpose (this is a great confusing-trace exhibit in Opik):

In [ ]:
from job_scout.graph import get_compiled_graph

graph = get_compiled_graph()
config = {"configurable": {"thread_id": thread_id}}

# BUG: a "new search" that forgets to null selected_job_id on the used thread
nodes = []
for chunk in graph.stream({"profile": profile, "cv_text": cv_text}, config=config, stream_mode="updates"):
    nodes.extend(chunk.keys())
print("nodes that ran:", nodes, " ← routed into tailor, not the search!")

In [ ]:
# THE FIX (what stream_search always does): explicit selected_job_id=None
nodes = []
for chunk in graph.stream({"profile": profile, "cv_text": cv_text, "selected_job_id": None},
                          config=config, stream_mode="updates"):
    nodes.extend(chunk.keys())
print("nodes that ran:", sorted(set(nodes)))

## 8. Render the tailored CV (LaTeX → PDF)

One ATS-friendly template; every user-derived value is LaTeX-escaped. With `tectonic`
installed you get a PDF; without it, the `.tex` plus an Overleaf pointer — rendering
never fails a run.

In [ ]:
import tempfile
from pathlib import Path as P
from job_scout.renderer import render_pdf

out = render_pdf(pack.cv, profile.name or "Candidate", P(tempfile.mkdtemp(prefix="job_scout_nb_")))
print("tex:", out.tex_path)
print("pdf:", out.pdf_path or f"(no PDF — {out.message})")

## 9. The evaluation stack

Built in teaching order (see `docs/opik_setup.md` §5–6 for the UI side):

1. **`extraction-cases`** — hand-labeled ground truth (`data/labels/expected_profiles.yaml`,
   scaffolded by `scripts/build_extraction_dataset.py --scaffold`, **corrected by a human**,
   pushed with `--push`). Scored by the deterministic `ProfileFieldAccuracy` — zero judges.
2. **`ranking-cases` / `tailoring-cases`** — exported from real traces with provenance
   (`scripts/build_eval_dataset.py`).
3. **The 5-metric suite** (`scripts/run_evals.py --suite …`): ProfileFieldAccuracy +
   FabricationRate (verifiable) · Hallucination + AnswerRelevance + FitExplanationQuality
   (judgments). Run one suite with two judge models and compare the experiments in Opik.
4. **Trajectory metrics** (`--trajectory`) — scores written back onto the traces.
5. **Online rules** — configured in the Opik UI (docs §5), including the honest finding that
   online rules cannot read the attached CV **PDF**; they judge the extracted `cv_text`.
6. **Calibration + annotation queue** — `scripts/setup_annotation_queue.py`, then
   `run_evals.py --calibration` reports judge-vs-human agreement, whatever it says.

In [ ]:
from job_scout.evals.metrics import ProfileFieldAccuracy

# The eval on-ramp, in miniature: deterministic scoring against a label.
metric = ProfileFieldAccuracy(track=False)
score = metric.score(
    output=profile.model_dump(),
    expected={"seniority": "senior", "primary_roles": ["Machine Learning Engineer"],
              "skills": ["python", "pytorch", "kubernetes"], "years_experience": 8.0,
              "locations": ["Berlin, Germany"], "languages": ["English", "German"], "remote_ok": True},
)
print(f"{score.value:.3f} — {score.reason}")
for field, value in score.metadata["per_field"].items():
    print(f"  {field:16s} {value:.2f}")

## 10. Where to look in Opik

- The **thread** for this notebook run: one extract trace, one search trace, one tailor
  trace (no fetch/rank spans), plus the deliberately-confusing §7 trace.
- **Datasets**: `job-scout-extraction-cases`, `job-scout-ranking-cases`,
  `job-scout-tailoring-cases` — items carry provenance back to their source traces.
- **Experiments**: compare the two ranking-judge runs side by side.
- **Annotation queue**: `job-scout-phase2-review`.

Weaknesses observed this phase are documented — **not fixed** — in
`docs/phase2_findings.md`. Fixing is Phase 3's story.

In [ ]:
from job_scout.tracing import opik_url
print("Opik project:", opik_url())